# Multi-Tool Business Assistant

This notebook demonstrates how to build an AI assistant that can use tools to interact with a database and provide information about product prices and stock levels. The assistant uses OpenAI's function calling feature to decide when to use tools and Gradio for the user interface.

In [30]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

## Environment Setup

Import necessary libraries and load environment variables for API access.

In [31]:
load_dotenv(override=True)
api_key = os.getenv("OPENROUTER_API_KEY")

client = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

MODEL = "openrouter/auto"

## OpenAI Client Configuration

Set up the OpenAI client using OpenRouter as the base URL for accessing various AI models.

In [45]:
system_message = """You are a helpful business assistant.

You can:
- Get product prices using get_price
- Check product stock using check_stock

Use the appropriate tool based on user query.
"""

## System Prompt

Define the AI's role and capabilities. This tells the model what it can do and how to behave.

In [33]:
conn = sqlite3.connect("business.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
    name TEXT PRIMARY KEY,
    price INTEGER,
    stock INTEGER
)
""")

## Database Initialization

Create a SQLite database table to store product information including names, prices, and stock levels.

In [34]:
cursor.execute("INSERT OR IGNORE INTO products VALUES ('whey', 2000, 50)")
cursor.execute("INSERT OR IGNORE INTO products VALUES ('creatine', 1500, 30)")
cursor.execute("INSERT OR IGNORE INTO products VALUES ('oats', 100, 100)")
conn.commit()


## Sample Data Insertion

Add some sample products to the database for testing purposes.

In [35]:
def get_price(item):
    cursor.execute("SELECT price FROM products WHERE name=?", (item.lower(),))
    result = cursor.fetchone()
    
    if result:
        return f"Price of {item} is ₹{result[0]}"
    return f"{item} not found"

## Tool Functions

Define the functions that the AI can call to retrieve product information from the database.

In [36]:
get_price("whey")

'Price of whey is ₹2000'

In [37]:
def get_stock(item):
    cursor.execute("SELECT stock FROM products WHERE name=?", (item.lower(),))
    result = cursor.fetchone()
    
    if result:
        return f"Stock of {item} is {result[0]}"
    return f"{item} not found"

In [39]:
get_stock("creatine")

'Stock of creatine is 30'

## Function Testing

Quick tests to verify that the tool functions work correctly with the database.

In [ ]:
price_function = {
    "name": "get_price",
    "description": "Get price of a product",
    "parameters": {
        "type": "object",
        "properties": {
            "item": {"type": "string"}
        },
        "required": ["item"]
    }
}

# tools = [{"type": "function", "function": price_function}]

In [41]:
stock_function = {
    "name": "get_stock",
    "description": "Get stock of a product",
    "parameters": { 
        "type": "object",
        "properties": {
            "item": {"type": "string"}
        },
        "required": ["item"]
    }
}

## Tool Schemas

Define the JSON schemas that describe each tool's name, description, and parameters to the AI model.

In [42]:
tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": stock_function}
]

## Tools Configuration

Combine the individual tool schemas into a list that will be passed to the OpenAI API.

In [43]:
tools

[{'type': 'function',
  'function': {'name': 'get_price',
   'description': 'Get price of a product',
   'parameters': {'type': 'object',
    'properties': {'item': {'type': 'string'}},
    'required': ['item']}}},
 {'type': 'function',
  'function': {'name': 'get_stock',
   'description': 'Get stock of a product',
   'parameters': {'type': 'object',
    'properties': {'item': {'type': 'string'}},
    'required': ['item']}}}]

In [54]:
def handle_tool_calls(message):
    print("\n🔧 Handling tool calls...")

    tool_messages = []

    for tool_call in message.tool_calls:
        print(f"\n📦 Tool Requested: {tool_call.function.name}")
        print(f"📥 Raw Arguments: {tool_call.function.arguments}")

        args = json.loads(tool_call.function.arguments)

        if tool_call.function.name == "get_price":
            result = get_price(args["item"])

        elif tool_call.function.name == "get_stock":
            result = get_stock(args["item"])

        tool_messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result
        })

    print("✅ Tool execution completed\n")
    return tool_messages

## Tool Call Handler

Function to process tool calls from the AI model, execute the corresponding functions, and format the results.

In [50]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}]

    for h in history:
        messages.append({"role": h["role"], "content": h["content"]})

    messages.append({"role": "user", "content": message})

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    msg = response.choices[0].message

    # 🔥 Handle tool calls properly
    if msg.tool_calls:
        messages.append(msg)

        tool_responses = handle_tool_calls(msg)

        for tr in tool_responses:
            messages.append(tr)

        final_response = client.chat.completions.create(
            model=MODEL,
            messages=messages
        )

        return final_response.choices[0].message.content

    return msg.content

## Main Chat Function

The core function that handles user messages, manages conversation history, calls the AI model, and coordinates tool usage.

In [29]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.



🔧 Handling tool calls...

📦 Tool Requested: get_stock
📥 Raw Arguments: {"item":"creatin"}
✅ Tool execution completed


🔧 Handling tool calls...

📦 Tool Requested: get_stock
📥 Raw Arguments: {"item":"Creatine"}
✅ Tool execution completed


🔧 Handling tool calls...

📦 Tool Requested: get_price
📥 Raw Arguments: {"item":"Creatine"}
✅ Tool execution completed



## Launch the Application

Start the Gradio web interface to interact with the AI business assistant and full code display.

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

# -------------------------
# ENV SETUP
# -------------------------
load_dotenv(override=True)
api_key = os.getenv("OPENROUTER_API_KEY")

print("\n🔐 Checking API Key...")
if api_key:
    print(f"✅ API Key loaded: {api_key[:8]}********")
else:
    print("❌ API Key NOT found")

# -------------------------
# OPENROUTER CLIENT
# -------------------------
client = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

MODEL = "openrouter/auto"

# -------------------------
# SYSTEM PROMPT
# -------------------------
system_message = """You are a helpful business assistant.
You can answer questions about product prices.

If needed, use the get_price tool to fetch accurate prices.
"""

# -------------------------
# DATABASE SETUP
# -------------------------
print("\n🗄️ Setting up database...")

conn = sqlite3.connect("business.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
    name TEXT PRIMARY KEY,
    price INTEGER,
    stock INTEGER
)
""")

cursor.execute("INSERT OR IGNORE INTO products VALUES ('whey', 2000, 50)")
cursor.execute("INSERT OR IGNORE INTO products VALUES ('creatine', 1500, 30)")
cursor.execute("INSERT OR IGNORE INTO products VALUES ('oats', 100, 100)")
conn.commit()

print("✅ Database ready with sample data")

# -------------------------
# TOOL: GET PRICE
# -------------------------
def get_price(item):
    print("\n🛠️ TOOL EXECUTION STARTED: get_price")
    print(f"👉 Input Item: {item}")

    cursor.execute("SELECT price FROM products WHERE name=?", (item.lower(),))
    result = cursor.fetchone()

    if result:
        print(f"✅ Price Found: ₹{result[0]}")
        return f"Price of {item} is ₹{result[0]}"
    
    print("❌ Item not found in DB")
    return f"{item} not found"

# -------------------------
# TOOL SCHEMA
# -------------------------
price_function = {
    "name": "get_price",
    "description": "Get price of a product",
    "parameters": {
        "type": "object",
        "properties": {
            "item": {
                "type": "string",
                "description": "Product name"
            }
        },
        "required": ["item"]
    }
}

tools = [{"type": "function", "function": price_function}]

# -------------------------
# HANDLE TOOL CALLS
# -------------------------
def handle_tool_calls(message):
    print("\n🔧 Handling tool calls...")

    tool_messages = []

    for tool_call in message.tool_calls:
        print(f"\n📦 Tool Requested: {tool_call.function.name}")
        print(f"📥 Raw Arguments: {tool_call.function.arguments}")

        args = json.loads(tool_call.function.arguments)

        if tool_call.function.name == "get_price":
            result = get_price(args["item"])

            tool_messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })

    print("✅ Tool execution completed\n")
    return tool_messages

# -------------------------
# CHAT FUNCTION
# -------------------------
def chat(message, history):
    print("\n" + "="*50)
    print(f"🧠 USER INPUT: {message}")

    messages = [{"role": "system", "content": system_message}]

    for h in history:
        messages.append({"role": h["role"], "content": h["content"]})

    messages.append({"role": "user", "content": message})

    print("\n📡 Sending request to LLM...")

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    msg = response.choices[0].message

    # -------------------------
    # TOOL CALL FLOW
    # -------------------------
    if msg.tool_calls:
        print("\n🤖 LLM DECIDED TO USE TOOL")

        for tc in msg.tool_calls:
            print(f"🔹 Tool Name: {tc.function.name}")
            print(f"🔹 Arguments: {tc.function.arguments}")

        messages.append(msg)

        tool_responses = handle_tool_calls(msg)

        for tr in tool_responses:
            messages.append(tr)

        print("\n📡 Sending tool result back to LLM...")

        final_response = client.chat.completions.create(
            model=MODEL,
            messages=messages
        )

        final_text = final_response.choices[0].message.content

        print(f"\n💬 FINAL RESPONSE: {final_text}")
        print("="*50 + "\n")

        return final_text

    # -------------------------
    # NORMAL RESPONSE
    # -------------------------
    print("\n💬 LLM DIRECT RESPONSE (no tool used)")
    print(f"👉 {msg.content}")
    print("="*50 + "\n")

    return msg.content

# -------------------------
# GRADIO UI
# -------------------------
print("\n🚀 Launching Gradio App...\n")

gr.ChatInterface(
    fn=chat,
    type="messages",
    title="🧰 Business AI Assistant"
).launch()


🔐 Checking API Key...
✅ API Key loaded: sk-or-v1********

🗄️ Setting up database...
✅ Database ready with sample data

🚀 Launching Gradio App...

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.



🧠 USER INPUT: hii

📡 Sending request to LLM...

💬 LLM DIRECT RESPONSE (no tool used)
👉 Hi there! How can I help you today? 



🧠 USER INPUT: what is price of whey?

📡 Sending request to LLM...

🤖 LLM DECIDED TO USE TOOL
🔹 Tool Name: get_price
🔹 Arguments: {"item":"whey"}

🔧 Handling tool calls...

📦 Tool Requested: get_price
📥 Raw Arguments: {"item":"whey"}

🛠️ TOOL EXECUTION STARTED: get_price
👉 Input Item: whey
✅ Price Found: ₹2000
✅ Tool execution completed


📡 Sending tool result back to LLM...

💬 FINAL RESPONSE: I found that the price of whey is ₹2000.


🧠 USER INPUT: of bcaa?

📡 Sending request to LLM...

🤖 LLM DECIDED TO USE TOOL
🔹 Tool Name: get_price
🔹 Arguments: {"item":"bcaa"}

🔧 Handling tool calls...

📦 Tool Requested: get_price
📥 Raw Arguments: {"item":"bcaa"}

🛠️ TOOL EXECUTION STARTED: get_price
👉 Input Item: bcaa
❌ Item not found in DB
✅ Tool execution completed


📡 Sending tool result back to LLM...

💬 FINAL RESPONSE: I couldn't find "bcaa." Could you please specify 